# MLB Payroll Efficiency: Data Cleaning

This notebook prepares MLB team payroll and performance data for SQL analysis and Tableau. It keeps ten full seasons and creates a clean team-season dataset.

## 1. Import Libraries

Pandas is used to clean and analyze the data. Path helps define the input and output file locations.

In [1]:
from pathlib import Path

import pandas as pd

## 2. Load Data

Load the original payroll dataset. The raw CSV remains unchanged.

In [2]:
raw_data_path = Path("../data/raw/mlb_payrolls_raw.csv")

df = pd.read_csv(raw_data_path)
df.head()

,Team,Team Name,Year,Average Age,Total Payroll Allocations,Active 26-Man,Injured,Retained,Buried,Wins,Losses,Postseason
0,OAK,Oakland Athletics,2024,26.5,"$62,132,581","$28,956,713","$15,581,092","$15,557,073","$1,763,221",69,93,No Playoffs
1,PIT,Pittsburgh Pirates,2024,27.7,"$84,050,989","$51,220,210","$14,524,211","$15,341,351","$2,965,217",76,86,No Playoffs
2,TB,Tampa Bay Rays,2024,26.8,"$89,707,422","$37,691,876","$13,179,262","$34,675,167","$1,706,572",80,82,No Playoffs
3,DET,Detroit Tigers,2024,26.0,"$96,961,614","$33,226,992","$26,677,166","$36,920,494","$1,070,295",86,76,Wildcard
4,CLE,Cleveland Guardians,2024,26.3,"$105,224,582","$50,885,032","$21,120,833","$22,945,837","$10,272,880",92,69,Division Winner


## 3. Inspect Data

Review the dataset size, column names, and data types before cleaning.

In [3]:
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())
print("Data types:")
print(df.dtypes)

Rows and columns: (420, 12)
Column names: ['Team', 'Team Name', 'Year', 'Average Age', 'Total Payroll Allocations', 'Active 26-Man', 'Injured', 'Retained', 'Buried', 'Wins', 'Losses', 'Postseason']
Data types:
Team                             str
Team Name                        str
Year                           int64
Average Age                  float64
Total Payroll Allocations        str
Active 26-Man                    str
Injured                          str
Retained                         str
Buried                           str
Wins                           int64
Losses                         int64
Postseason                       str
dtype: object


## 4. Keep Needed Data

Keep the seven source columns needed for the analysis. Use ten full seasons and exclude the shortened 2020 season.

In [4]:
columns_to_keep = [
    "Team",
    "Team Name",
    "Year",
    "Total Payroll Allocations",
    "Wins",
    "Losses",
    "Postseason"
]

seasons_to_keep = [
    2014, 2015, 2016, 2017, 2018,
    2019, 2021, 2022, 2023, 2024
]

df_clean = df[df["Year"].isin(seasons_to_keep)].copy()
df_clean = df_clean[columns_to_keep].copy()

df_clean.shape

(300, 7)

## 5. Clean Data

Rename the columns so they are consistent and easy to use. Convert payroll from currency text into whole dollars.

In [5]:
df_clean = df_clean.rename(columns={
    "Team": "team_abbreviation",
    "Team Name": "team_name",
    "Year": "season",
    "Total Payroll Allocations": "payroll",
    "Wins": "wins",
    "Losses": "losses",
    "Postseason": "postseason_result"
})

df_clean.columns.tolist()

['team_abbreviation',
 'team_name',
 'season',
 'payroll',
 'wins',
 'losses',
 'postseason_result']

In [6]:
df_clean["payroll"] = df_clean["payroll"].str.replace("$", "", regex=False)
df_clean["payroll"] = df_clean["payroll"].str.replace(",", "", regex=False)
df_clean["payroll"] = df_clean["payroll"].astype("int64")

print(df_clean["payroll"].dtype)
df_clean[["payroll"]].head()

int64


,payroll
0,62132581
1,84050989
2,89707422
3,96961614
4,105224582


## 6. Check Data Quality

Check for missing values, duplicate team-seasons, and unexpected season or team-name problems before creating new fields.

In [7]:
print("Missing values:")
print(df_clean.isna().sum())

duplicate_team_seasons = df_clean.duplicated(
    subset=["team_abbreviation", "season"]
).sum()

print("Duplicate team-seasons:", duplicate_team_seasons)

Missing values:
team_abbreviation    0
team_name            0
season               0
payroll              0
wins                 0
losses               0
postseason_result    0
dtype: int64
Duplicate team-seasons: 0


In [8]:
print("Rows per season:")
print(df_clean.groupby("season").size())

print("Postseason categories:")
print(df_clean["postseason_result"].value_counts())

Rows per season:
season
2014    30
2015    30
2016    30
2017    30
2018    30
2019    30
2021    30
2022    30
2023    30
2024    30
dtype: int64
Postseason categories:
postseason_result
No Playoffs        196
Division Winner     60
Wildcard            44
Name: count, dtype: int64


In [9]:
team_name_counts = df_clean.groupby("team_abbreviation")["team_name"].nunique()
teams_with_multiple_names = team_name_counts[team_name_counts > 1]

print("Number of team abbreviations:", df_clean["team_abbreviation"].nunique())
print("Abbreviations with multiple team names:")
print(teams_with_multiple_names)

Number of team abbreviations: 30
Abbreviations with multiple team names:
Series([], Name: team_name, dtype: int64)


## 7. Create Calculated Fields

Create seven fields used in the SQL analysis and Tableau dashboard. Calculations use exact payroll dollars before the final results are rounded.

In [10]:
df_clean["games_played"] = df_clean["wins"] + df_clean["losses"]

df_clean["win_percentage"] = (
    df_clean["wins"] / df_clean["games_played"]
).round(3)

df_clean["playoff_qualified"] = (
    df_clean["postseason_result"] != "No Playoffs"
).astype(int)

df_clean[["games_played", "win_percentage", "playoff_qualified"]].head()

,games_played,win_percentage,playoff_qualified
0,162,0.426,0
1,162,0.469,0
2,162,0.494,0
3,162,0.531,1
4,161,0.571,1


In [11]:
exact_payroll_millions = df_clean["payroll"] / 1_000_000
league_average_payroll = df_clean.groupby("season")["payroll"].transform("mean")

df_clean["payroll_millions"] = exact_payroll_millions.round(2)

df_clean["league_avg_payroll_millions"] = (
    league_average_payroll / 1_000_000
).round(2)

df_clean["payroll_vs_league_avg_pct"] = (
    df_clean["payroll"] / league_average_payroll * 100
).round(1)

df_clean["wins_per_million"] = (
    df_clean["wins"] / exact_payroll_millions
).round(3)

df_clean[[
    "team_name",
    "season",
    "payroll_millions",
    "league_avg_payroll_millions",
    "payroll_vs_league_avg_pct",
    "wins_per_million"
]].head()

,team_name,season,payroll_millions,league_avg_payroll_millions,payroll_vs_league_avg_pct,wins_per_million
0,Oakland Athletics,2024,62.13,166.6,37.3,1.111
1,Pittsburgh Pirates,2024,84.05,166.6,50.5,0.904
2,Tampa Bay Rays,2024,89.71,166.6,53.8,0.892
3,Detroit Tigers,2024,96.96,166.6,58.2,0.887
4,Cleveland Guardians,2024,105.22,166.6,63.2,0.874


## 8. Check the Final Dataset

Sort the data and confirm its final size, columns, and quality before analysis and export.

In [12]:
df_clean = df_clean.sort_values(["season", "team_name"])
df_clean = df_clean.reset_index(drop=True)

print("Final rows and columns:", df_clean.shape)
print("Final missing values:", df_clean.isna().sum().sum())
print("Final duplicate team-seasons:", df_clean.duplicated(
    subset=["team_abbreviation", "season"]
).sum())
print("Final columns:", df_clean.columns.tolist())

Final rows and columns: (300, 14)
Final missing values: 0
Final duplicate team-seasons: 0
Final columns: ['team_abbreviation', 'team_name', 'season', 'payroll', 'wins', 'losses', 'postseason_result', 'games_played', 'win_percentage', 'playoff_qualified', 'payroll_millions', 'league_avg_payroll_millions', 'payroll_vs_league_avg_pct', 'wins_per_million']


In [13]:
df_clean[[
    "payroll",
    "wins",
    "losses",
    "games_played",
    "win_percentage",
    "wins_per_million"
]].describe().round(2)

,payroll,wins,losses,games_played,win_percentage,wins_per_million
count,300.0,300.00,300.00,300.00,300.00,300.00
mean,140761619.7,80.98,80.98,161.96,0.50,0.66
std,55850035.4,12.73,12.71,0.26,0.08,0.26
min,42421870.0,41.00,51.00,161.00,0.25,0.22
25%,97225111.0,72.00,72.00,162.00,0.45,0.46
50%,132378237.5,81.50,80.00,162.00,0.50,0.60
75%,174483181.0,90.00,89.25,162.00,0.56,0.79
max,341673777.0,111.00,121.00,163.00,0.68,1.85


## 9. Explore Results

Calculate two simple summaries that connect the cleaned data to the main SQL and Tableau findings.

In [14]:
league_average_for_grouping = df_clean.groupby("season")["payroll"].transform("mean")

at_or_above_average = df_clean[
    df_clean["payroll"] >= league_average_for_grouping
]

below_average = df_clean[
    df_clean["payroll"] < league_average_for_grouping
]

print("At or above league-average payroll:")
print("Team-seasons:", len(at_or_above_average))
print("Average wins:", round(at_or_above_average["wins"].mean(), 1))
print("Postseason rate:", round(at_or_above_average["playoff_qualified"].mean() * 100, 1), "%")

print("Below league-average payroll:")
print("Team-seasons:", len(below_average))
print("Average wins:", round(below_average["wins"].mean(), 1))
print("Postseason rate:", round(below_average["playoff_qualified"].mean() * 100, 1), "%")

At or above league-average payroll:
Team-seasons: 142
Average wins: 86.1
Postseason rate: 47.9 %
Below league-average payroll:
Team-seasons: 158
Average wins: 76.4
Postseason rate: 22.8 %


In [15]:
average_payroll_by_season = df_clean.groupby("season")["payroll"].mean()
average_payroll_by_season = (average_payroll_by_season / 1_000_000).round(2)

average_payroll_by_season

season
2014    119.15
2015    127.59
2016    133.15
2017    139.78
2018    138.38
2019    137.69
2021    131.08
2022    149.79
2023    164.41
2024    166.60
Name: payroll, dtype: float64

## 10. Export Data

Create the processed-data folder if needed and export the final CSV without an extra index column.

In [16]:
processed_data_folder = Path("../data/processed")
processed_data_folder.mkdir(parents=True, exist_ok=True)

output_path = processed_data_folder / "mlb_payroll_efficiency_clean.csv"
df_clean.to_csv(output_path, index=False)

print("Exported file:", output_path)
df_clean.head()

Exported file: ../data/processed/mlb_payroll_efficiency_clean.csv


,team_abbreviation,team_name,season,payroll,wins,losses,postseason_result,games_played,win_percentage,playoff_qualified,payroll_millions,league_avg_payroll_millions,payroll_vs_league_avg_pct,wins_per_million
0,ARI,Arizona Diamondbacks,2014,106343190,64,98,No Playoffs,162,0.395,0,106.34,119.15,89.2,0.602
1,ATL,Atlanta Braves,2014,116865707,79,83,No Playoffs,162,0.488,0,116.87,119.15,98.1,0.676
2,BAL,Baltimore Orioles,2014,113562943,96,66,Division Winner,162,0.593,1,113.56,119.15,95.3,0.845
3,BOS,Boston Red Sox,2014,170095758,71,91,No Playoffs,162,0.438,0,170.10,119.15,142.8,0.417
4,CHC,Chicago Cubs,2014,90364741,73,89,No Playoffs,162,0.451,0,90.36,119.15,75.8,0.808
